# MATH 250 — Traffic Sign Classification
#### By Siling Guo, Megan Joseph, An Truong, Anna Wadlow, Robert Yav

**Prerequisites:** `math250_preprocessing.ipynb` must have been run first  
and its output files must be in the shared Google Drive folder.

---
### Instructions for each member
1. Run **Sections 1–4** (setup, load, split, eval helper) — identical for everyone.
2. Run **only your assigned section** in Section 5:

| Member | Section | Dim Reduction |
|--------|---------|---------------|
| 1 | 5A | PCA |
| 2 | 5B | UMAP |
| 3 | 5C | t-SNE |

3. Run **Section 6** (results summary) after your section.

---
### Notebook structure
| Section | Description |
|---------|-------------|
| 1 | Setup & Imports |
| 2 | Mount Drive & load preprocessed data |
| 3 | Train / Test Split |
| 4 | Evaluation helper + shared plots |
| 5A | PCA + 4 classifiers |
| 5B | UMAP + 4 classifiers |
| 5C | t-SNE + 4 classifiers |
| 6 | Results summary table |

---
## Section 1 — Setup & Imports

In [7]:
!pip install xgboost umap-learn --quiet

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from google.colab import drive # use when running on cloud

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, f1_score,
    classification_report, confusion_matrix, ConfusionMatrixDisplay,
)
from xgboost import XGBClassifier
import umap
from imblearn.over_sampling import SMOTE

# ── Must match preprocessing notebook ─────────────────────────────────────────
TARGET_SIZE  = (64, 64)
RANDOM_STATE = 42
TEST_SIZE    = 0.20

np.random.seed(RANDOM_STATE)   # ensures identical random plots for everyone
print('Imports OK.')

Imports OK.


---
## Section 2 — Mount Drive & Load Preprocessed Data

Set `DRIVE_LOAD_PATH` to the same shared folder used in the preprocessing notebook.

In [8]:
drive.mount('/content/drive') # when running on cloud

# ── Set this to the shared folder (same as preprocessing notebook) ────────────
DRIVE_LOAD_PATH = '/content/drive/Shareddrives/MATH 250: Traffic Sign Classification' # when running on cloud
#DRIVE_LOAD_PATH = r'C:\Users\andtr\Downloads\MATH 250\project files\preprocessed_files'

# ── Load feature matrix, labels, and class names ──────────────────────────────
X       = np.load(os.path.join(DRIVE_LOAD_PATH, 'X.npy'))
y       = np.load(os.path.join(DRIVE_LOAD_PATH, 'y.npy'))
classes = np.load(os.path.join(DRIVE_LOAD_PATH, 'le_classes.npy'), allow_pickle=True)
meta_df = pd.read_csv(os.path.join(DRIVE_LOAD_PATH, 'crop_metadata.csv'))

# Reconstruct LabelEncoder from saved classes
le = LabelEncoder()
le.classes_ = classes

print(f'X shape    : {X.shape}')       # (N_samples, 4096)
print(f'y shape    : {y.shape}')
print(f'Classes    : {len(classes)}  -> {classes}')
print(f'Meta rows  : {len(meta_df)}')

Mounted at /content/drive
X shape    : (29512, 4096)
y shape    : (29512,)
Classes    : 46  -> ['addedLane' 'curveLeft' 'curveRight' 'dip' 'doNotEnter' 'doNotPass'
 'intersection' 'keepRight' 'laneEnds' 'merge' 'noLeftTurn' 'noRightTurn'
 'pedestrianCrossing' 'rampSpeedAdvisory20' 'rampSpeedAdvisory35'
 'rampSpeedAdvisory45' 'rampSpeedAdvisory50' 'rampSpeedAdvisoryUrdbl'
 'rightLaneMustTurn' 'roundabout' 'school' 'schoolSpeedLimit25'
 'signalAhead' 'slow' 'speedLimit15' 'speedLimit25' 'speedLimit30'
 'speedLimit35' 'speedLimit40' 'speedLimit45' 'speedLimit50'
 'speedLimit55' 'speedLimit65' 'speedLimitUrdbl' 'stop' 'stopAhead'
 'thruMergeLeft' 'thruMergeRight' 'thruTrafficMergeLeft'
 'truckSpeedLimit55' 'turnLeft' 'turnRight' 'yield' 'yieldAhead'
 'zoneAhead25' 'zoneAhead45']
Meta rows  : 29512


---
## Section 3 — Train / Test Split

**Use `RANDOM_STATE=42` and `stratify=y` here.**  
Because everyone loads the same `X` and `y` from Drive and uses the same seed,  
all members get **identical** train/test splits even in separate Colab sessions.

In [9]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size    = TEST_SIZE,
    random_state = RANDOM_STATE,
    stratify     = y
)

print(f'Training : {X_train.shape[0]:,} samples')
print(f'Test     : {X_test.shape[0]:,} samples')
print(f'Features : {X_train.shape[1]:,}')

Training : 23,609 samples
Test     : 5,903 samples
Features : 4,096


In [10]:
# Apply SMOTE only to training data to handle imbalanced data. Set seed for reproducibility.
sm = SMOTE(random_state = 42)
X_train, y_train = sm.fit_resample(X_train, y_train)

---
## Section 4 — Shared Plots & Evaluation Helper

Run these cells once.  The plots are loaded from the PNGs saved to Drive  
during preprocessing — everyone sees the exact same figures.

In [11]:
# ── Display shared preprocessing plots from Drive ─────────────────────────────
plot_files = [
    ('plot_class_distribution.png',    'Class Distribution (before augmentation)'),
    ('plot_class_dist_augmented.png',  'Class Distribution (after augmentation)'),
    ('plot_sample_grid.png',           'Sample Grid — one crop per class'),
    ('plot_augmentation_examples.png', 'Augmentation Examples'),
    ('plot_crop_size_dist.png',        'Original Crop Size Distribution'),
]

for fname, title in plot_files:
    fpath = os.path.join(DRIVE_LOAD_PATH, fname)
    if os.path.exists(fpath):
        img = plt.imread(fpath)
        fig, ax = plt.subplots(figsize=(14, 5))
        ax.imshow(img)
        ax.axis('off')
        ax.set_title(title, fontsize=12)
        plt.tight_layout()
        plt.show()
    else:
        print(f'  [missing] {fname} — run preprocessing notebook first')

  [missing] plot_class_distribution.png — run preprocessing notebook first
  [missing] plot_class_dist_augmented.png — run preprocessing notebook first
  [missing] plot_sample_grid.png — run preprocessing notebook first
  [missing] plot_augmentation_examples.png — run preprocessing notebook first
  [missing] plot_crop_size_dist.png — run preprocessing notebook first


In [12]:
# ── Evaluation helper ─────────────────────────────────────────────────────────
def evaluate_model(model_name, y_true, y_pred, class_names=None, plot_cm=True,
                   save_path=None):
    """
    Compute accuracy, balanced accuracy, and macro F1-score.
    Optionally display and save a normalised confusion matrix.

    Parameters
    ----------
    model_name  : str   — shown in output header
    y_true      : array — ground-truth integer labels
    y_pred      : array — predicted integer labels
    class_names : list  — human-readable strings (pass le.classes_)
    plot_cm     : bool  — display the confusion matrix
    save_path   : str   — if given, save the confusion matrix PNG here

    Returns dict with keys 'accuracy', 'balanced_accuracy', 'f1_macro'
    """
    acc     = accuracy_score(y_true, y_pred)
    bal_acc = balanced_accuracy_score(y_true, y_pred)
    f1      = f1_score(y_true, y_pred, average='macro', zero_division=0)

    sep = '=' * 58
    print(f'\n{sep}')
    print(f'  {model_name}')
    print(sep)
    print(f'  Accuracy          : {acc:.4f}')
    print(f'  Balanced Accuracy : {bal_acc:.4f}  <- primary metric')
    print(f'  Macro F1-Score    : {f1:.4f}')
    print()
    print(classification_report(y_true, y_pred,
                                target_names=class_names, zero_division=0))

    if plot_cm:
        cm  = confusion_matrix(y_true, y_pred, normalize='true')
        fig, ax = plt.subplots(figsize=(14, 12))
        ConfusionMatrixDisplay(cm, display_labels=class_names).plot(
            ax=ax, xticks_rotation=45, colorbar=True, cmap='Blues'
        )
        ax.set_title(f'Normalised Confusion Matrix\n{model_name}')
        plt.tight_layout()
        if save_path:
            plt.savefig(save_path, dpi=150)
            print(f'  Saved CM: {save_path}')
        plt.show()

    return {'model': model_name, 'accuracy': acc,
            'balanced_accuracy': bal_acc, 'f1_macro': f1}


print('evaluate_model() ready.')

evaluate_model() ready.


---
## Section 5B — UMAP + Classifiers

UMAP (Uniform Manifold Approximation and Projection) is a non-linear
dimension reduction method. Unlike t-SNE it **does** have a `.transform()`
method, so the workflow is the same as PCA: fit on the training set,
transform both train and test.

Two key hyperparameters:
- `n_neighbors` — controls how much local vs global structure is preserved.
  Larger values → more global structure captured.
- `min_dist` — controls how tightly points are packed in the embedding.
  Smaller values → tighter clusters.

We use `n_components=15` for classification (more dimensions = more
information for the classifier) and also produce a 2-D embedding
for visualization.

In [ ]:
# ── Fit UMAP for classification (15 components) ───────────────────────────────
# n_components=15 keeps more structure than 2-D for the classifiers to work with.
# n_neighbors=15 is the default; increase for more global structure.
# min_dist=0.1 allows moderately tight clusters.
N_UMAP_COMPONENTS = 15

reducer = umap.UMAP(
    n_components = N_UMAP_COMPONENTS,
    n_neighbors  = 15,
    min_dist     = 0.1,
    random_state = RANDOM_STATE
)
X_train_umap = reducer.fit_transform(X_train)   # fit on train only
X_test_umap  = reducer.transform(X_test)         # apply same transform to test

print(f'X_train_umap : {X_train_umap.shape}')
print(f'X_test_umap  : {X_test_umap.shape}')

# ── Also fit a separate 2-D reducer just for visualization ────────────────────
print('Fitting 2-D UMAP for visualization...')
reducer_2d   = umap.UMAP(n_components=2, n_neighbors=15, min_dist=0.1,
                          random_state=RANDOM_STATE)
X_train_2d   = reducer_2d.fit_transform(X_train)

fig, ax = plt.subplots(figsize=(14, 10))
scatter = ax.scatter(
    X_train_2d[:, 0], X_train_2d[:, 1],
    c=y_train, cmap='tab20', s=4, alpha=0.55
)
cbar = plt.colorbar(scatter, ax=ax, ticks=range(len(le.classes_)))
cbar.ax.set_yticklabels(le.classes_, fontsize=7)
ax.set_title('UMAP 2-D Embedding — Training Set')
ax.set_xlabel('UMAP Dimension 1')
ax.set_ylabel('UMAP Dimension 2')
plt.tight_layout()
plt.savefig(os.path.join(DRIVE_LOAD_PATH, 'plot_umap_embedding.png'), dpi=150)
plt.show()
print('Saved: plot_umap_embedding.png')

/usr/local/lib/python3.12/dist-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


In [ ]:
# ── UMAP + SVM ────────────────────────────────────────────────────────────────
svm_umap = SVC(kernel='rbf', C=10, gamma='scale', random_state=RANDOM_STATE)
svm_umap.fit(X_train_umap, y_train)
r_svm_umap = evaluate_model(
    'UMAP + SVM', y_test, svm_umap.predict(X_test_umap),
    class_names=le.classes_,
    save_path=os.path.join(DRIVE_LOAD_PATH, 'cm_umap_svm.png')
)

In [ ]:
# ── UMAP + Random Forest ──────────────────────────────────────────────────────
rf_umap = RandomForestClassifier(n_estimators=200, random_state=RANDOM_STATE, n_jobs=-1)
rf_umap.fit(X_train_umap, y_train)
r_rf_umap = evaluate_model(
    'UMAP + Random Forest', y_test, rf_umap.predict(X_test_umap),
    class_names=le.classes_,
    save_path=os.path.join(DRIVE_LOAD_PATH, 'cm_umap_rf.png')
)

In [ ]:
# ── UMAP + Naive Bayes ────────────────────────────────────────────────────────
nb_umap = GaussianNB()
nb_umap.fit(X_train_umap, y_train)
r_nb_umap = evaluate_model(
    'UMAP + Naive Bayes', y_test, nb_umap.predict(X_test_umap),
    class_names=le.classes_,
    save_path=os.path.join(DRIVE_LOAD_PATH, 'cm_umap_nb.png')
)

In [ ]:
# ── UMAP + XGBoost ────────────────────────────────────────────────────────────
xgb_umap = XGBClassifier(
    n_estimators=200, eval_metric='mlogloss',
    random_state=RANDOM_STATE, n_jobs=-1
)
xgb_umap.fit(X_train_umap, y_train)
r_xgb_umap = evaluate_model(
    'UMAP + XGBoost', y_test, xgb_umap.predict(X_test_umap),
    class_names=le.classes_,
    save_path=os.path.join(DRIVE_LOAD_PATH, 'cm_umap_xgb.png')
)

umap_results = [r_svm_umap, r_rf_umap, r_nb_umap, r_xgb_umap]

---
## Section 6 — Results Summary

Combine whichever result lists are available in this session into a ranked table.  
The table and bar chart are saved to Drive so the full team can compare results.

In [ ]:
# ── Collect whichever results exist in this session ───────────────────────────
all_results = []
for var_name in ['pca_results', 'umap_results', 'tsne_results']:
    if var_name in dir():
        all_results.extend(eval(var_name))

if not all_results:
    print('No results collected yet — run a section in 5A, 5B, or 5C first.')
else:
    summary = pd.DataFrame(all_results).sort_values('balanced_accuracy', ascending=False)
    summary = summary.rename(columns={
        'accuracy': 'Accuracy',
        'balanced_accuracy': 'Balanced Accuracy',
        'f1_macro': 'Macro F1'
    })

    print(summary[['model','Accuracy','Balanced Accuracy','Macro F1']].to_string(index=False))

    # ── Bar chart ─────────────────────────────────────────────────────────────
    fig, ax = plt.subplots(figsize=(14, 6))
    x     = np.arange(len(summary))
    width = 0.28
    ax.bar(x - width, summary['Accuracy'],          width, label='Accuracy',          color='steelblue')
    ax.bar(x,         summary['Balanced Accuracy'], width, label='Balanced Accuracy', color='coral')
    ax.bar(x + width, summary['Macro F1'],          width, label='Macro F1',          color='seagreen')
    ax.set_xticks(x)
    ax.set_xticklabels(summary['model'], rotation=40, ha='right', fontsize=9)
    ax.set_ylabel('Score')
    ax.set_ylim(0, 1.05)
    ax.set_title('Model Comparison — All Metrics')
    ax.legend()
    plt.tight_layout()
    plt.savefig(os.path.join(DRIVE_LOAD_PATH, 'plot_results_summary.png'), dpi=150)
    plt.show()
    print('Saved: plot_results_summary.png')

    # Save CSV
    out_csv = os.path.join(DRIVE_LOAD_PATH, 'results_summary.csv')
    summary.to_csv(out_csv, index=False)
    print(f'Saved: results_summary.csv')